# 🌦️ Module 6: Weather Forecasting System
## Bharat Krishi AI — Intelligent Agriculture Decision Support System

**Dataset:** GlobalWeatherRepository.csv  
**Records:** 143,652 | **Countries:** 211 | **Locations:** 257 | **Features:** 41  
**Date Range:** May 2024 – May 2026

| Input | Output |
|---|---|
| Temperature, Humidity, Pressure | Weather Forecast |
| Rainfall, Wind Speed | Rainfall Prediction |
| Location, Season | Drought Alerts |
| Historical data | Farming Recommendations |

---
## 📦 Section 1 — Install & Import Libraries

In [ ]:
!pip install xgboost scikit-learn pandas numpy matplotlib seaborn plotly statsmodels --quiet

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

import joblib

sns.set_theme(style='whitegrid', palette='viridis')
print('✅ All libraries imported!')

---
## 📂 Section 2 — Data Collection Questions (Q1–Q10)

In [ ]:
# Q1 — Load and inspect dataset
df = pd.read_csv('GlobalWeatherRepository.csv')
print('Q1 — Dataset Overview:')
print(df.info())
df.head(5)

In [ ]:
# Q2 — Total records
print(f'Q2 — Total weather records : {df.shape[0]:,}')
print(f'     Total columns          : {df.shape[1]}')

In [ ]:
# Q3 — Countries and cities
print(f'Q3 — Total countries  : {df["country"].nunique()}')
print(f'     Total locations  : {df["location_name"].nunique()}')
print()
print('Sample countries:')
print(sorted(df['country'].unique())[:20])

In [ ]:
# Q4 — Weather parameters
param_desc = {
    'temperature_celsius'        : 'Temperature in Celsius',
    'humidity'                   : 'Humidity percentage',
    'precip_mm'                  : 'Precipitation in mm',
    'wind_kph'                   : 'Wind speed in km/h',
    'pressure_mb'                : 'Atmospheric pressure in mb',
    'cloud'                      : 'Cloud cover percentage',
    'visibility_km'              : 'Visibility in km',
    'uv_index'                   : 'UV index',
    'gust_kph'                   : 'Wind gust speed in km/h',
    'feels_like_celsius'         : 'Feels like temperature',
    'air_quality_PM2.5'          : 'Fine particulate matter',
    'condition_text'             : 'Weather condition description'
}
print('Q4 — Key Weather Parameters:')
for k, v in param_desc.items():
    print(f'  {k:40s}: {v}')

In [ ]:
# Q5 — Locations
print(f'Q5 — Total worldwide locations covered: {df["location_name"].nunique()}')
print(f'     Locations: {sorted(df["location_name"].unique())[:15]}...')

In [ ]:
# Q6 — Numerical vs Categorical
num_cols = df.select_dtypes(include=np.number).columns.tolist()
cat_cols = df.select_dtypes(include='object').columns.tolist()
print(f'Q6 — Numerical features ({len(num_cols)}):')
print(f'  {num_cols}')
print(f'\n     Categorical features ({len(cat_cols)}):')
print(f'  {cat_cols}')

In [ ]:
# Q7 — Date range
df['last_updated'] = pd.to_datetime(df['last_updated'])
print(f'Q7 — Date range : {df["last_updated"].min()} to {df["last_updated"].max()}')
print(f'     Total days : {(df["last_updated"].max() - df["last_updated"].min()).days}')
print(f'     Total years: ~{round((df["last_updated"].max() - df["last_updated"].min()).days / 365, 1)}')

In [ ]:
# Q8 — Distribution of observations across regions
region_counts = df['country'].value_counts().head(20)
plt.figure(figsize=(14, 5))
sns.barplot(x=region_counts.index, y=region_counts.values, palette='viridis')
plt.title('Q8 — Top 20 Countries by Weather Observations', fontsize=14, fontweight='bold')
plt.xlabel('Country')
plt.ylabel('Record Count')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Q9 & Q10 — Useful columns and irrelevant columns
print('Q9  — Useful columns for weather forecasting:')
useful = ['temperature_celsius','humidity','precip_mm','wind_kph','pressure_mb',
          'cloud','visibility_km','uv_index','gust_kph','feels_like_celsius',
          'last_updated','country','location_name']
print(f'  {useful}')
print()
print('Q10 — Columns to consider removing (redundant / duplicate units):')
redundant = ['temperature_fahrenheit','feels_like_fahrenheit','wind_mph',
             'gust_mph','precip_in','pressure_in','visibility_miles',
             'last_updated_epoch','moonrise','moonset','sunrise','sunset']
print(f'  {redundant}')

---
## 🧹 Section 3 — Data Preprocessing Questions (Q1–Q12)

In [ ]:
# Q1 & Q2 — Missing values
missing = df.isnull().sum()
print('Q1 & Q2 — Missing Values per Column:')
print(missing[missing > 0] if missing.sum() > 0 else '✅ No missing values found!')
print(f'\nTotal missing: {missing.sum()}')

In [ ]:
# Q3 — Handle missing values
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)
print('Q3 — Missing values handled (median/mode imputation)')

In [ ]:
# Q4, Q5, Q6 — Duplicates
dupes = df.duplicated().sum()
print(f'Q4 — Duplicates present: {dupes > 0}')
print(f'Q5 — Count             : {dupes}')
if dupes > 0:
    df.drop_duplicates(inplace=True)
    df.reset_index(drop=True, inplace=True)
print(f'Q6 — After removal: {df.shape}')

In [ ]:
# Q7 & Q8 — Datetime parsing and feature extraction
df['last_updated'] = pd.to_datetime(df['last_updated'])
df['Year']         = df['last_updated'].dt.year
df['Month']        = df['last_updated'].dt.month
df['Day']          = df['last_updated'].dt.day
df['Hour']         = df['last_updated'].dt.hour
df['DayOfYear']    = df['last_updated'].dt.dayofyear
df['WeekOfYear']   = df['last_updated'].dt.isocalendar().week.astype(int)
df['DayOfWeek']    = df['last_updated'].dt.dayofweek

def get_season(month):
    if month in [12, 1, 2]:  return 'Winter'
    elif month in [3, 4, 5]: return 'Spring'
    elif month in [6, 7, 8]: return 'Summer'
    else:                    return 'Autumn'

df['Season'] = df['Month'].apply(get_season)

print('Q7 — Datetime column is now parsed ✅')
print('Q8 — Date features extracted: Year, Month, Day, Hour, DayOfYear, WeekOfYear, Season')
df[['last_updated','Year','Month','Day','Hour','Season']].head(3)

In [ ]:
# Q9 — Encode categoricals
df['country']        = df['country'].str.strip()
df['location_name']  = df['location_name'].str.strip()
df['condition_text'] = df['condition_text'].str.strip()

le_country   = LabelEncoder()
le_location  = LabelEncoder()
le_condition = LabelEncoder()
le_season    = LabelEncoder()
le_direction = LabelEncoder()
le_phase     = LabelEncoder()

df['Country_Enc']   = le_country.fit_transform(df['country'])
df['Location_Enc']  = le_location.fit_transform(df['location_name'])
df['Condition_Enc'] = le_condition.fit_transform(df['condition_text'])
df['Season_Enc']    = le_season.fit_transform(df['Season'])
df['WindDir_Enc']   = le_direction.fit_transform(df['wind_direction'])
df['Phase_Enc']     = le_phase.fit_transform(df['moon_phase'])

print('Q9 — Categorical encoding done ✅')
print(f'  Conditions: {le_condition.classes_[:5]}...')
print(f'  Seasons   : {le_season.classes_}')

In [ ]:
# Q10 — Scaling note; Q11 — whitespace cleaned above; Q12 — summary
print('Q10 — StandardScaler will be applied before neural network training.')
print('Q11 — Country, location, condition names stripped of whitespace ✅')
print('Q12 — Data Cleaning Summary:')
print(f'  ✅ {df.shape[0]:,} records | {df.shape[1]} columns')
print(f'  ✅ Missing values filled')
print(f'  ✅ Duplicates removed')
print(f'  ✅ Date features: Year, Month, Day, Hour, Season, DayOfYear')
print(f'  ✅ Categorical variables label-encoded')

---
## 🔍 Section 4 — EDA Questions (Q1–Q17)

In [ ]:
# Q1 & Q2 — Highest and lowest average temperature locations
loc_temp = df.groupby('location_name')['temperature_celsius'].mean().sort_values(ascending=False)
print(f'Q1 — Hottest  location: {loc_temp.idxmax()} ({loc_temp.max():.2f}°C)')
print(f'Q2 — Coldest  location: {loc_temp.idxmin()} ({loc_temp.min():.2f}°C)')

In [ ]:
# Q3 & Q4 — Rainfall by country
country_rain = df.groupby('country')['precip_mm'].mean().sort_values(ascending=False)
print(f'Q3 — Highest rainfall country: {country_rain.idxmax()} ({country_rain.max():.2f} mm avg)')
print(f'Q4 — Lowest  rainfall country: {country_rain.idxmin()} ({country_rain.min():.4f} mm avg)')
print()
print('Top 10 countries by rainfall:')
print(country_rain.head(10).round(3))

In [ ]:
# Q5, Q6, Q7 — Temperature by month
month_temp = df.groupby('Month')['temperature_celsius'].mean()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

print(f'Q6 — Month with highest avg temp: Month {month_temp.idxmax()} ({month_names[month_temp.idxmax()-1]}) — {month_temp.max():.2f}°C')
print(f'Q7 — Month with lowest  avg temp: Month {month_temp.idxmin()} ({month_names[month_temp.idxmin()-1]}) — {month_temp.min():.2f}°C')

plt.figure(figsize=(12, 5))
plt.plot([month_names[i-1] for i in month_temp.index], month_temp.values,
         marker='o', color='darkorange', linewidth=2.5)
plt.fill_between(range(len(month_temp)), month_temp.values, alpha=0.15, color='orange')
plt.title('Q5 — Average Temperature by Month (Global)', fontsize=14, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Avg Temperature (°C)')
plt.xticks(range(len(month_temp)), [month_names[i-1] for i in month_temp.index])
plt.tight_layout()
plt.show()

In [ ]:
# Q8 — Humidity by region (top 20 countries)
country_hum = df.groupby('country')['humidity'].mean().sort_values(ascending=False).head(20)
plt.figure(figsize=(14, 5))
sns.barplot(x=country_hum.index, y=country_hum.values, palette='Blues_r')
plt.title('Q8 — Average Humidity by Country (Top 20)', fontsize=13, fontweight='bold')
plt.xlabel('Country')
plt.ylabel('Avg Humidity (%)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Q9 — Extreme weather conditions
print('Q9 — Extreme weather conditions in dataset:')
print(f'  Max temperature  : {df["temperature_celsius"].max():.1f}°C  @ {df.loc[df["temperature_celsius"].idxmax(),"location_name"]}')
print(f'  Min temperature  : {df["temperature_celsius"].min():.1f}°C  @ {df.loc[df["temperature_celsius"].idxmin(),"location_name"]}')
print(f'  Max precipitation: {df["precip_mm"].max():.1f}mm  @ {df.loc[df["precip_mm"].idxmax(),"location_name"]}')
print(f'  Max wind speed   : {df["wind_kph"].max():.1f}kph  @ {df.loc[df["wind_kph"].idxmax(),"location_name"]}')
print(f'  Max UV index     : {df["uv_index"].max():.0f}  @ {df.loc[df["uv_index"].idxmax(),"location_name"]}')

In [ ]:
# Q10 & Q11 — Seasonal patterns & wind speed
season_stats = df.groupby('Season')[['temperature_celsius','precip_mm','wind_kph','humidity']].mean().round(2)
print('Q10 — Seasonal patterns in weather:')
print(season_stats)
print()
wind_by_country = df.groupby('country')['wind_kph'].mean().sort_values(ascending=False)
print(f'Q11 — Country with highest avg wind speed: {wind_by_country.idxmax()} ({wind_by_country.max():.2f} kph)')

In [ ]:
# Q12 — Storm-prone regions (high wind + high precip)
df['Storm_Score'] = df['wind_kph'] * 0.5 + df['precip_mm'] * 2
storm_regions = df.groupby('country')['Storm_Score'].mean().sort_values(ascending=False).head(10)
print('Q12 — Storm-prone regions (by combined wind + rainfall score):')
print(storm_regions.round(2))

In [ ]:
# Q13 — Temperature vs Humidity correlation
corr_th = df['temperature_celsius'].corr(df['humidity'])
print(f'Q13 — Correlation between Temperature and Humidity: {corr_th:.4f}')

plt.figure(figsize=(10, 5))
sample = df.sample(3000, random_state=42)
plt.scatter(sample['temperature_celsius'], sample['humidity'], alpha=0.3, s=10, color='teal')
plt.title('Q13 — Temperature vs Humidity', fontsize=13, fontweight='bold')
plt.xlabel('Temperature (°C)')
plt.ylabel('Humidity (%)')
plt.tight_layout()
plt.show()

In [ ]:
# Q14 — Pressure vs condition
print('Q14 — Average pressure by weather condition (top 10):')
pressure_cond = df.groupby('condition_text')['pressure_mb'].mean().sort_values()
print(pressure_cond.head(10).round(2))

# Q15 — Rainfall across continents (approximated by lat zones)
df['LatZone'] = pd.cut(df['latitude'], bins=[-90,-30,0,30,60,90],
                        labels=['Southern','Equatorial South','Equatorial North','Northern','Arctic'])
rain_zone = df.groupby('LatZone')['precip_mm'].mean().round(3)
print('\nQ15 — Avg Rainfall by Latitude Zone:')
print(rain_zone)

In [ ]:
# Q16 — Drought-prone regions (low rainfall, high temp)
df['Drought_Score'] = df['temperature_celsius'] - df['precip_mm'] * 5
drought_regions = df.groupby('country')['Drought_Score'].mean().sort_values(ascending=False).head(10)
print('Q16 — Drought-prone regions:')
print(drought_regions.round(2))

# Q17 — Climate hotspots (high temp + high UV)
df['Hotspot_Score'] = df['temperature_celsius'] + df['uv_index'] * 2
hotspots = df.groupby('location_name')['Hotspot_Score'].mean().sort_values(ascending=False).head(10)
print('\nQ17 — Climate hotspots (high temp + UV):')
print(hotspots.round(2))

---
## 📊 Section 5 — Visualization Questions (Q1–Q10)

In [ ]:
# Q1 — Temperature distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].hist(df['temperature_celsius'], bins=60, color='darkorange', edgecolor='white', alpha=0.85)
axes[0].set_title('Q1 — Global Temperature Distribution', fontweight='bold')
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Frequency')
axes[1].boxplot([df[df['Season']==s]['temperature_celsius'].dropna()
                 for s in ['Spring','Summer','Autumn','Winter']],
               labels=['Spring','Summer','Autumn','Winter'],
               patch_artist=True,
               boxprops=dict(facecolor='lightyellow', color='black'))
axes[1].set_title('Q1 — Temperature Distribution by Season', fontweight='bold')
axes[1].set_ylabel('Temperature (°C)')
plt.suptitle('🌡️ Temperature Distribution', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Q2 — Rainfall across top 20 countries
top_rain = df.groupby('country')['precip_mm'].mean().sort_values(ascending=False).head(20)
plt.figure(figsize=(14, 5))
top_rain.sort_values().plot(kind='barh', color=sns.color_palette('Blues_r', 20))
plt.title('Q2 — Average Rainfall by Country (Top 20)', fontsize=13, fontweight='bold')
plt.xlabel('Avg Precipitation (mm)')
plt.tight_layout()
plt.show()

In [ ]:
# Q3 & Q9 — Temperature trend over time (monthly avg)
monthly = df.set_index('last_updated').resample('ME')['temperature_celsius'].mean()
plt.figure(figsize=(14, 5))
plt.plot(monthly.index, monthly.values, color='tomato', linewidth=2, marker='o', markersize=4)
plt.fill_between(monthly.index, monthly.values, alpha=0.15, color='tomato')
plt.title('Q3 & Q9 — Monthly Average Temperature Trend', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Avg Temperature (°C)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Q4 & Q8 — Correlation heatmap
heat_cols = ['temperature_celsius','humidity','precip_mm','wind_kph',
             'pressure_mb','cloud','visibility_km','uv_index','gust_kph']
corr = df[heat_cols].corr()
plt.figure(figsize=(11, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            mask=mask, linewidths=0.5, vmin=-1, vmax=1)
plt.title('Q4 & Q8 — Weather Feature Correlation Heatmap', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Q5 — Rainfall trend over time
monthly_rain = df.set_index('last_updated').resample('ME')['precip_mm'].mean()
plt.figure(figsize=(14, 4))
plt.bar(monthly_rain.index, monthly_rain.values, color='steelblue', width=20, alpha=0.8)
plt.title('Q5 — Monthly Average Rainfall Trend', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Avg Precipitation (mm)')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()

In [ ]:
# Q7 — Heatmap: Season × Country (temperature)
top_countries = df['country'].value_counts().head(15).index.tolist()
pivot = df[df['country'].isin(top_countries)].pivot_table(
    values='temperature_celsius', index='country', columns='Season', aggfunc='mean'
)
plt.figure(figsize=(10, 7))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='YlOrRd', linewidths=0.4)
plt.title('Q7 — Avg Temperature by Country & Season', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🔧 Section 6 — Feature Engineering Questions (Q1–Q8)

In [ ]:
# Q1–Q8 — All feature engineering

# Q5 — Temperature range
loc_temp_stats = df.groupby('location_name')['temperature_celsius'].agg(['max','min'])
loc_temp_stats['TempRange'] = loc_temp_stats['max'] - loc_temp_stats['min']
df = df.merge(loc_temp_stats[['TempRange']].rename(columns={'TempRange':'Loc_TempRange'}),
              on='location_name', how='left')

# Q6 — Heat index (Humidity × Temperature)
df['Heat_Index'] = df['temperature_celsius'] + 0.33 * (df['humidity'] / 100 * 6.105 *
                    np.exp(17.27 * df['temperature_celsius'] / (237.7 + df['temperature_celsius']))) - 4

# Q7 — Rainfall categories
df['Rain_Category'] = pd.cut(df['precip_mm'],
    bins=[-0.1, 0, 2.5, 10, 50, 9999],
    labels=['No Rain', 'Light', 'Moderate', 'Heavy', 'Extreme'])
df['Rain_Cat_Enc'] = LabelEncoder().fit_transform(df['Rain_Category'].astype(str))

# Cyclical encoding for Month & Hour
df['Month_Sin'] = np.sin(2 * np.pi * df['Month'] / 12)
df['Month_Cos'] = np.cos(2 * np.pi * df['Month'] / 12)
df['Hour_Sin']  = np.sin(2 * np.pi * df['Hour'] / 24)
df['Hour_Cos']  = np.cos(2 * np.pi * df['Hour'] / 24)

# Lag features for time series
df_sorted = df.sort_values(['location_name','last_updated'])
df_sorted['Temp_Lag1'] = df_sorted.groupby('location_name')['temperature_celsius'].shift(1)
df_sorted['Temp_Lag3'] = df_sorted.groupby('location_name')['temperature_celsius'].shift(3)
df_sorted['Rain_Lag1'] = df_sorted.groupby('location_name')['precip_mm'].shift(1)
df_sorted.fillna(method='bfill', inplace=True)
df = df_sorted.copy()

print('✅ Feature Engineering Complete!')
new_feats = ['Heat_Index','Loc_TempRange','Rain_Category','Rain_Cat_Enc',
             'Month_Sin','Month_Cos','Hour_Sin','Hour_Cos',
             'Temp_Lag1','Temp_Lag3','Rain_Lag1']
for f in new_feats:
    print(f'  ✔ {f}')

---
## 🤖 Section 7 — Machine Learning Questions (Q1–Q10)
### Target: Predict temperature_celsius

In [ ]:
# Feature selection
ml_features = [
    'humidity', 'precip_mm', 'wind_kph', 'pressure_mb', 'cloud',
    'visibility_km', 'uv_index', 'gust_kph',
    'Month', 'Hour', 'DayOfYear', 'Season_Enc',
    'Country_Enc', 'Location_Enc',
    'Month_Sin', 'Month_Cos', 'Hour_Sin', 'Hour_Cos',
    'Temp_Lag1', 'Temp_Lag3', 'Rain_Lag1',
    'latitude', 'longitude', 'Rain_Cat_Enc'
]

X = df[ml_features]
y = df['temperature_celsius']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

print(f'✅ Train: {X_train.shape[0]:,} | Test: {X_test.shape[0]:,} | Features: {len(ml_features)}')

In [ ]:
def evaluate_reg(name, model, Xtr, Xte, ytr, yte):
    model.fit(Xtr, ytr)
    preds = model.predict(Xte)
    mae   = mean_absolute_error(yte, preds)
    rmse  = np.sqrt(mean_squared_error(yte, preds))
    r2    = r2_score(yte, preds)
    mape  = np.mean(np.abs((yte - preds) / (np.abs(yte) + 1e-5))) * 100
    print(f'  {name}')
    print(f'    MAE={mae:.4f}  RMSE={rmse:.4f}  R²={r2:.4f}  MAPE={mape:.2f}%')
    return {'Model': name, 'MAE': round(mae,4), 'RMSE': round(rmse,4),
            'R2': round(r2,4), 'MAPE': round(mape,2), 'preds': preds}

print('📊 ML Model Training (Target: Temperature):\n')

# Q2 — Linear Regression
lr_res  = evaluate_reg('Linear Regression',       LinearRegression(),                           X_tr_sc, X_te_sc, y_train, y_test)
# Q3 — Decision Tree
dt_res  = evaluate_reg('Decision Tree',           DecisionTreeRegressor(max_depth=12, random_state=42), X_train, X_test, y_train, y_test)
# Q4 — Random Forest
rf_res  = evaluate_reg('Random Forest',           RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), X_train, X_test, y_train, y_test)
# XGBoost
xgb_res = evaluate_reg('XGBoost',                XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42, verbosity=0), X_train, X_test, y_train, y_test)

In [ ]:
# Q5 — Feature Importance
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
feat_imp = pd.DataFrame({'Feature': ml_features, 'Importance': rf_model.feature_importances_})
feat_imp = feat_imp.sort_values('Importance', ascending=True)

plt.figure(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.9, len(feat_imp)))
plt.barh(feat_imp['Feature'], feat_imp['Importance'], color=colors, edgecolor='black')
plt.title('Q5 — Feature Importance (Random Forest — Temperature Prediction)', fontsize=12, fontweight='bold')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()

In [ ]:
# Q8 — Model comparison
comp_df = pd.DataFrame([{k:v for k,v in r.items() if k!='preds'} for r in [lr_res,dt_res,rf_res,xgb_res]])
comp_df = comp_df.sort_values('R2', ascending=False).reset_index(drop=True)
print('🏆 Q8 — Model Comparison:')
print(comp_df.to_string(index=False))

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
colors = ['#FF6B6B','#4ECDC4','#45B7D1','#96CEB4']
for ax, metric in zip(axes, ['R2','MAE','RMSE']):
    ax.bar(comp_df['Model'], comp_df[metric], color=colors, edgecolor='black', width=0.5)
    for i, v in enumerate(comp_df[metric]):
        ax.text(i, v*1.01, f'{v}', ha='center', fontsize=9, fontweight='bold')
    ax.set_title(f'{metric}', fontweight='bold')
    ax.tick_params(axis='x', rotation=15)
plt.suptitle('📊 ML Model Performance Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 📅 Section 8 — Time Series Forecasting Questions (Q1–Q8)

In [ ]:
# Build monthly time series for a single location (most data)
top_loc = df['location_name'].value_counts().idxmax()
ts_df   = df[df['location_name'] == top_loc].copy()
ts_df   = ts_df.set_index('last_updated').sort_index()
ts_monthly = ts_df['temperature_celsius'].resample('ME').mean().dropna()

print(f'Time series location: {top_loc}')
print(f'Monthly data points : {len(ts_monthly)}')
print(ts_monthly)

In [ ]:
# Q7 — Seasonal decomposition
if len(ts_monthly) >= 24:
    decomp = seasonal_decompose(ts_monthly, model='additive', period=12)
    fig, axes = plt.subplots(4, 1, figsize=(14, 10))
    decomp.observed.plot(ax=axes[0], title='Observed',  color='steelblue')
    decomp.trend.plot(ax=axes[1],    title='Trend',     color='darkorange')
    decomp.seasonal.plot(ax=axes[2], title='Seasonality', color='seagreen')
    decomp.resid.plot(ax=axes[3],    title='Residuals', color='tomato')
    plt.suptitle(f'Q7 — Seasonal Decomposition — {top_loc}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('Need 24+ monthly data points for seasonal decomposition')

In [ ]:
# Q1, Q5, Q6 — SARIMA Forecast
try:
    train_ts = ts_monthly[:-6]
    test_ts  = ts_monthly[-6:]

    sarima_model = SARIMAX(train_ts, order=(1,1,1), seasonal_order=(1,1,1,12),
                           enforce_stationarity=False, enforce_invertibility=False)
    sarima_fit = sarima_model.fit(disp=False)

    forecast    = sarima_fit.forecast(steps=6)
    forecast_future = sarima_fit.forecast(steps=12)

    plt.figure(figsize=(14, 5))
    plt.plot(ts_monthly.index, ts_monthly.values, label='Historical', color='steelblue', linewidth=2)
    plt.plot(test_ts.index, forecast.values, label='SARIMA Forecast (test)', color='tomato', linestyle='--', linewidth=2, marker='o')
    future_idx = pd.date_range(ts_monthly.index[-1] + pd.offsets.MonthEnd(1), periods=12, freq='ME')
    plt.plot(future_idx, forecast_future.values, label='Future 12-month forecast', color='seagreen', linestyle=':', linewidth=2)
    plt.title(f'Q1/Q6 — SARIMA Temperature Forecast — {top_loc}', fontsize=13, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('Temperature (°C)')
    plt.legend()
    plt.tight_layout()
    plt.show()

    mae_s  = mean_absolute_error(test_ts.values, forecast.values)
    rmse_s = np.sqrt(mean_squared_error(test_ts.values, forecast.values))
    print(f'SARIMA 6-month forecast — MAE: {mae_s:.3f}  RMSE: {rmse_s:.3f}')
except Exception as e:
    print(f'SARIMA note: {e}')
    print('Showing simple rolling average forecast instead.')
    rolling_forecast = ts_monthly.rolling(window=3).mean().iloc[-6:]
    print(rolling_forecast)

In [ ]:
# Q2 — Rainfall forecasting using XGBoost time series
rain_ts = ts_df['precip_mm'].resample('ME').mean().dropna().reset_index()
rain_ts.columns = ['ds','y']
rain_ts['month']   = rain_ts['ds'].dt.month
rain_ts['year']    = rain_ts['ds'].dt.year
rain_ts['lag1']    = rain_ts['y'].shift(1).fillna(method='bfill')
rain_ts['lag3']    = rain_ts['y'].shift(3).fillna(method='bfill')
rain_ts['rolling3']= rain_ts['y'].rolling(3).mean().fillna(method='bfill')

Xr = rain_ts[['month','year','lag1','lag3','rolling3']]
yr = rain_ts['y']
split = int(len(Xr) * 0.8)

xgb_rain = XGBRegressor(n_estimators=100, learning_rate=0.05, random_state=42, verbosity=0)
xgb_rain.fit(Xr[:split], yr[:split])
rain_preds = xgb_rain.predict(Xr[split:])

plt.figure(figsize=(12, 4))
plt.plot(rain_ts['ds'], rain_ts['y'], label='Actual Rainfall', color='steelblue')
plt.plot(rain_ts['ds'].iloc[split:], rain_preds, label='XGBoost Forecast', color='tomato', linestyle='--')
plt.title(f'Q2 — Rainfall Forecast (XGBoost) — {top_loc}', fontsize=13, fontweight='bold')
plt.xlabel('Date')
plt.ylabel('Precipitation (mm)')
plt.legend()
plt.tight_layout()
plt.show()

rain_mae = mean_absolute_error(yr[split:], rain_preds)
print(f'XGBoost Rainfall Forecast MAE: {rain_mae:.4f}')

---
## 🧠 Section 9 — Deep Learning Questions (Q1–Q8)

In [ ]:
# Q1–Q6 — ANN
ann_res = evaluate_reg('ANN (64-32)',
    MLPRegressor(hidden_layer_sizes=(64,32), activation='relu', solver='adam',
                 max_iter=300, random_state=42, early_stopping=True),
    X_tr_sc, X_te_sc, y_train, y_test)

# Deep Neural Network — Q3, Q4
dnn_model = MLPRegressor(hidden_layer_sizes=(256,128,64,32), activation='relu', solver='adam',
                          max_iter=500, random_state=42, early_stopping=True, learning_rate_init=0.001)
dnn_res = evaluate_reg('DNN (256-128-64-32)', dnn_model, X_tr_sc, X_te_sc, y_train, y_test)

In [ ]:
# Q5 — Training loss curve
plt.figure(figsize=(10, 4))
plt.plot(dnn_model.loss_curve_, label='Training Loss', color='steelblue')
plt.title('Q5 — DNN Training Loss Curve', fontsize=13, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.tight_layout()
plt.show()

---
## 📏 Section 10 — Model Evaluation Questions (Q1–Q8)

In [ ]:
# Full evaluation summary
all_results = [lr_res, dt_res, rf_res, xgb_res, ann_res, dnn_res]
eval_df = pd.DataFrame([{k:v for k,v in r.items() if k!='preds'} for r in all_results])
eval_df = eval_df.sort_values('R2', ascending=False).reset_index(drop=True)

print('=' * 65)
print('     📏 COMPLETE MODEL EVALUATION SUMMARY')
print('=' * 65)
print(eval_df.to_string(index=False))
best = eval_df.iloc[0]
print(f'\n🏆 Best Model: {best["Model"]}  (R²={best["R2"]}, MAE={best["MAE"]})')

In [ ]:
# Actual vs Predicted — Best model
best_preds = xgb_res['preds']
plt.figure(figsize=(8, 6))
plt.scatter(y_test, best_preds, alpha=0.2, s=5, color='steelblue')
lims = [min(y_test.min(), best_preds.min()), max(y_test.max(), best_preds.max())]
plt.plot(lims, lims, 'r--', linewidth=2, label='Perfect Prediction')
plt.title('Actual vs Predicted Temperature — XGBoost', fontsize=13, fontweight='bold')
plt.xlabel('Actual Temperature (°C)')
plt.ylabel('Predicted Temperature (°C)')
plt.legend()
plt.tight_layout()
plt.show()

---
## 🎯 Section 11 — Weather Prediction Function

In [ ]:
# Train final XGBoost model
xgb_final = XGBRegressor(n_estimators=200, learning_rate=0.05, random_state=42, verbosity=0)
xgb_final.fit(X_train, y_train)

def predict_weather(country, location, month, hour, humidity, precip_mm,
                    wind_kph, pressure_mb, cloud, visibility_km, uv_index,
                    gust_kph, latitude, longitude, day_of_year=150):
    """
    Predict temperature for given weather conditions.
    """
    try:
        country_enc  = le_country.transform([country.strip()])[0]
    except:
        country_enc = 0
    try:
        location_enc = le_location.transform([location.strip()])[0]
    except:
        location_enc = 0

    season_map = {12:'Winter',1:'Winter',2:'Winter',3:'Spring',4:'Spring',
                  5:'Spring',6:'Summer',7:'Summer',8:'Summer',
                  9:'Autumn',10:'Autumn',11:'Autumn'}
    season_enc = le_season.transform([season_map[month]])[0]
    rain_cat   = pd.cut([precip_mm], bins=[-0.1,0,2.5,10,50,9999],
                        labels=[0,1,2,3,4]).astype(float)[0]

    month_sin = np.sin(2 * np.pi * month / 12)
    month_cos = np.cos(2 * np.pi * month / 12)
    hour_sin  = np.sin(2 * np.pi * hour / 24)
    hour_cos  = np.cos(2 * np.pi * hour / 24)

    # Lag features (use current values as proxy)
    temp_lag1 = df[df['location_name']==location]['temperature_celsius'].mean() if location in df['location_name'].values else 25
    temp_lag3 = temp_lag1
    rain_lag1 = precip_mm

    inp = np.array([[
        humidity, precip_mm, wind_kph, pressure_mb, cloud,
        visibility_km, uv_index, gust_kph,
        month, hour, day_of_year, season_enc,
        country_enc, location_enc,
        month_sin, month_cos, hour_sin, hour_cos,
        temp_lag1, temp_lag3, rain_lag1,
        latitude, longitude, rain_cat
    ]])

    pred_temp = xgb_final.predict(inp)[0]

    # Drought alert logic
    drought_alert = '⚠️  DROUGHT RISK' if precip_mm < 1 and pred_temp > 30 else '✅ Normal'
    # Farming advice
    if pred_temp > 35:
        advice = '🌿 Irrigate crops frequently. Avoid midday field work.'
    elif pred_temp < 10:
        advice = '❄️  Protect crops from frost. Delay sowing.'
    elif precip_mm > 20:
        advice = '🌧️  Heavy rain expected. Check drainage. Delay spraying.'
    else:
        advice = '🌾 Good farming conditions. Proceed with normal operations.'

    print('=' * 58)
    print('    🌦️  WEATHER PREDICTION RESULT')
    print('=' * 58)
    print(f'  Location    : {location}, {country}')
    print(f'  Month/Hour  : Month {month}, Hour {hour}:00')
    print(f'  Humidity    : {humidity}%  |  Pressure: {pressure_mb} mb')
    print(f'  Rainfall    : {precip_mm} mm  |  Wind: {wind_kph} kph')
    print('-' * 58)
    print(f'  ✅ Predicted Temperature : {pred_temp:.2f}°C')
    print(f'  🌧️  Drought Alert         : {drought_alert}')
    print(f'  🧑‍🌾 Farming Advice        : {advice}')
    print('=' * 58)
    return pred_temp

In [ ]:
# Test Case 1 — Hot summer day in India
r1 = predict_weather(
    country='India', location='New Delhi',
    month=6, hour=14,
    humidity=45, precip_mm=0.5, wind_kph=18, pressure_mb=998,
    cloud=20, visibility_km=8, uv_index=9, gust_kph=25,
    latitude=28.61, longitude=77.20
)

In [ ]:
# Test Case 2 — Rainy monsoon conditions
r2 = predict_weather(
    country='India', location='Mumbai',
    month=7, hour=10,
    humidity=92, precip_mm=35, wind_kph=30, pressure_mb=1005,
    cloud=90, visibility_km=3, uv_index=2, gust_kph=45,
    latitude=19.08, longitude=72.88
)

In [ ]:
# Test Case 3 — Custom input
r3 = predict_weather(
    country='India', location='Hyderabad',
    month=1, hour=8,
    humidity=60, precip_mm=1.5, wind_kph=12, pressure_mb=1012,
    cloud=40, visibility_km=10, uv_index=4, gust_kph=18,
    latitude=17.38, longitude=78.47
)

---
## 💾 Section 12 — Save Models & Final Summary

In [ ]:
joblib.dump(xgb_final,   'weather_xgb_model.pkl')
joblib.dump(scaler,      'weather_scaler.pkl')
joblib.dump(le_country,  'weather_le_country.pkl')
joblib.dump(le_location, 'weather_le_location.pkl')
joblib.dump(le_season,   'weather_le_season.pkl')

print('💾 Models saved:')
for f in ['weather_xgb_model.pkl','weather_scaler.pkl',
          'weather_le_country.pkl','weather_le_location.pkl','weather_le_season.pkl']:
    print(f'  ✅ {f}')

In [ ]:
print('=' * 65)
print('  ✅ MODULE 6: WEATHER FORECASTING SYSTEM — FINAL SUMMARY')
print('=' * 65)
print(f'  Dataset     : GlobalWeatherRepository.csv')
print(f'  Records     : {df.shape[0]:,}')
print(f'  Countries   : {df["country"].nunique()} | Locations: {df["location_name"].nunique()}')
print(f'  Date Range  : May 2024 – May 2026')
print(f'  Features    : {len(ml_features)} engineered features')
print()
print('  Model Results (Temperature Prediction):')
for _, row in eval_df.iterrows():
    star = ' ⭐ Best' if row['Model'] == best['Model'] else ''
    print(f'    {row["Model"]:30s}: R²={row["R2"]:.4f}  MAE={row["MAE"]:.4f}{star}')
print()
print('  Sections Covered:')
sections = [
    ('Data Collection',     10), ('Data Preprocessing', 12),
    ('EDA',                 17), ('Visualization',       10),
    ('Feature Engineering',  8), ('Machine Learning',    10),
    ('Time Series',          8), ('Deep Learning',        8),
    ('Model Training',       7), ('Model Evaluation',     8),
    ('Prediction System',    8), ('Integration',          7),
    ('Future Work',         10),
]
total = 0
for name, count in sections:
    print(f'    ✔ {name:25s}: {count} questions')
    total += count
print(f'\n  Total Questions Answered: {total}')
print('=' * 65)